# The Central Limit Theorem
Very few of the data histograms that we have seen in this course have been bell shaped. When we have come across a bell shaped distribution, it has almost invariably been an empirical histogram of a statistic based on a random sample.

The examples below show two very different situations in which an approximate bell shape appears in such histograms.

In [1]:
import math
import numpy as np
from datascience import make_array
import polars as pl
import hvplot.polars
from scipy import stats
path_data = '../../../assets/data/'

In [2]:
colors = pl.read_csv(path_data + 'roulette_wheel.csv').get_column('Color')
pockets = make_array('0','00')
for i in np.arange(1, 37):
    pockets = np.append(pockets, str(i))

wheel = pl.DataFrame([
    pl.Series('Pocket', pockets),
    colors,
])

## Net Gain in Roulette
In an earlier section, the bell appeared as the rough shape of the total amount of money we would make if we placed the same bet repeatedly on different spins of a roulette wheel. 

In [3]:
wheel

Pocket,Color
str,str
"""0""","""green"""
"""00""","""green"""
"""1""","""red"""
"""2""","""black"""
"""3""","""red"""
…,…
"""32""","""red"""
"""33""","""black"""
"""34""","""red"""


Recall that the bet on red pays even money, 1 to 1. We defined the function `red_winnings` that returns the net winnings on one \$1 bet on red. Specifically, the function takes a color as its argument and returns 1 if the color is red. For all other colors it returns -1.

In [4]:
def red_winnings(color):
    if color == 'red':
        return 1
    else:
        return -1

The DataFrame `red` shows each pocket's winnings on red.

In [5]:
# POLARS_CHANGE: map_elements calls red_winnings on each color and needs a return dtype
red = wheel.with_columns(
    wheel.get_column('Color').map_elements(red_winnings, return_dtype=int).alias('Winnings: Red')
    )
red

Pocket,Color,Winnings: Red
str,str,i64
"""0""","""green""",-1
"""00""","""green""",-1
"""1""","""red""",1
"""2""","""black""",-1
"""3""","""red""",1
…,…,…
"""32""","""red""",1
"""33""","""black""",-1
"""34""","""red""",1


Your net gain on one bet is one random draw from the `Winnings: Red` column. There is an 18/38 chance making \$1, and a 20/38 chance of making -\$1. This probability distribution is shown in the histogram below.

In [6]:
red.select('Winnings: Red').hvplot.hist('Winnings: Red', bins=np.arange(-1.5, 1.6, 1), normed=True)

Histogram with 'Winnings: Red' on the x-axis and 'Percent per uniit' on the y-axis. Two bars are shown. The bar centered at x=-1 has a height just above 50. The bar centered at x=1 has a height just below 50.

Now suppose you bet many times on red. Your net winnings will be the sum of many draws made at random with replacement from the distribution above.

It will take a bit of math to list all the possible values of your net winnings along with all of their chances. We won't do that; instead, we will approximate the probability distribution by simulation, as we have done all along in this course. 

The code below simulates your net gain if you bet \$1 on red on 400 different spins of the roulette wheel. 

In [7]:
num_bets = 400
repetitions = 10000

net_gain_red = make_array()

for i in np.arange(repetitions):
    spins = red.sample(num_bets, with_replacement=True)
    new_net_gain_red = spins.get_column('Winnings: Red').sum()
    net_gain_red = np.append(net_gain_red, new_net_gain_red)


results = pl.DataFrame([
    pl.Series('Net Gain on Red', net_gain_red)
    ])

In [8]:
results.hvplot.hist('Net Gain on Red', bins=np.arange(-80, 50, 6), normed=True)

Histogram with 'Net Gain on Red' on the x-axis and 'Percent per unit' on the y-axis. The data is symmetrical and bell shaped with a peak about x=-20 and ranging from x=-80 to x=40.

That's a roughly bell shaped histogram, even though the distribution we are drawing from is nowhere near bell shaped.

**Center.** The distribution is centered near -20 dollars, roughly. To see why, note that your winnings will be \$1 on about 18/38 of the bets, and -$1 on the remaining 20/38. So your average winnings per dollar bet will be roughly -5.26 cents:

In [9]:
average_per_bet = 1*(18/38) + (-1)*(20/38)
average_per_bet

-0.05263157894736842

So in 400 bets you expect that your net gain will be about -\$21:

In [10]:
400 * average_per_bet

-21.052631578947366

For confirmation, we can compute the mean of the 10,000 simulated net gains:

In [11]:
results.to_series(0).mean()

-21.0242

**Spread.** Run your eye along the curve starting at the center and notice that the point of inflection is near 0. On a bell shaped curve, the SD is the distance from the center to a point of inflection. The center is roughly -\$20, which means that the SD of the distribution is around \$20.

In the next section we will see where the \$20 comes from. For now, let's confirm our observation by simply calculating the SD of the 10,000 simulated net gains:

In [12]:
# POLARS_CHANGE: ddof=0 matches np.std (population SD)
results.to_series(0).std(ddof=0)

20.150717465142527

**Summary.** The net gain in 400 bets is the sum of the 400 amounts won on each individual bet. The probability distribution of that sum is approximately normal, with an average and an SD that we can approximate.

## Average Flight Delay
The DataFrame `united` contains data on departure delays of 13,825 United Airlines domestic flights out of San Francisco airport in the summer of 2015. As we have seen before, the distribution of delays has a long right-hand tail.

In [13]:
united = pl.read_csv(path_data + 'united_summer2015.csv')

In [14]:
united.select('Delay').hvplot.hist('Delay', bins=np.arange(-20, 300, 10), normed=True)

Histogram with 'Delay' on the x-axis and 'Percent per unit' on the y-axis. The histogram has the tallest bar on the left centered just to the left of x=0 with a dramatic decrease in height until the bars become so small they're not visible until about x=150, however the graph extends to x=300.

The mean delay was about 16.6 minutes and the SD was about 39.5 minutes. Notice how large the SD is, compared to the mean. Those large deviations on the right have an effect, even though they are a very small proportion of the data.

In [15]:
mean_delay = united.get_column('Delay').mean()
sd_delay = united.get_column('Delay').std(ddof=0)

mean_delay, sd_delay

(16.658155515370705, 39.480199851609314)

Now suppose we sampled 400 delays at random with replacement. You could sample without replacement if you like, but the results would be very similar to with-replacement sampling. If you sample a few hundred out of 13,825 without replacement, you hardly change the population each time you pull out a value.

In the sample, what could the average delay be? We expect it to be around 16 or 17, because that's the population average; but it is likely to be somewhat off. Let's see what we get by sampling. We'll work with the DataFrame `delay` that only contains the column of delays.

In [16]:
delay = united.select('Delay')

In [17]:
delay.sample(400, with_replacement=True).get_column('Delay').mean()

15.81

The sample average varies according to how the sample comes out, so we will simulate the sampling process repeatedly and draw the empirical histogram of the sample average. That will be an approximation to the probability histogram of the sample average.

In [18]:
sample_size = 400
repetitions = 10000

means = make_array()

for i in np.arange(repetitions):
    sample = delay.sample(sample_size, with_replacement=True)
    new_mean = sample.get_column('Delay').mean()
    means = np.append(means, new_mean)

results = pl.DataFrame([
    pl.Series('Sample Mean', means)
])

In [19]:
results.hvplot.hist('Sample Mean', bins=np.arange(10, 25, 0.5), normed=True)

Histogram with 'Sample Mean' on the x-axis and 'Percent per unit' on the y-axis. The x axis ranges from x=10 to x=24. The graph is mostly symmetric about approx x=16 with a slight tail extending to the right.

Once again, we see a rough bell shape, even though we are drawing from a very skewed distribution. The bell is centered somewhere between 16 ad 17, as we expect.

## Central Limit Theorem

The reason why the bell shape appears in such settings is a remarkable result of probability theory called the **Central Limit Theorem**. 

**The Central Limit Theorem says that the probability distribution of the sum or average of a large random sample drawn with replacement will be roughly normal, *regardless of the distribution of the population from which the sample is drawn*.**

As we noted when we were studying Chebychev's bounds, results that can be applied to random samples *regardless of the distribution of the population* are very powerful, because in data science we rarely know the distribution of the population.

The Central Limit Theorem makes it possible to make inferences with very little knowledge about the population, provided we have a large random sample. That is why it is central to the field of statistical inference.

## Proportion of Purple Flowers
Recall Mendel's probability model for the colors of the flowers of a species of pea plant. The model says that the flower colors of the plants are like draws made at random with replacement from {Purple, Purple, Purple, White}.

In a large sample of plants, about what proportion will have purple flowers? We would expect the answer to be about 0.75, the proportion purple in the model. And, because proportions are means, the Central Limit Theorem says that the distribution of the sample proportion of purple plants is roughly normal.

We can confirm this by simulation. Let's simulate the proportion of purple-flowered plants in a sample of 200 plants.

In [20]:
colors = make_array('Purple', 'Purple', 'Purple', 'White')

model = pl.DataFrame([pl.Series('Color', colors)])

model

Color
str
"""Purple"""
"""Purple"""
"""Purple"""
"""White"""


In [21]:
props = make_array()

num_plants = 200
repetitions = 10000

for i in np.arange(repetitions):
    sample = model.sample(num_plants, with_replacement=True)
    new_prop = (sample.get_column('Color') == 'Purple').sum()/num_plants
    props = np.append(props, new_prop)

results = pl.DataFrame([pl.Series('Sample Proportion: 200', props)])

In [22]:
results.hvplot.hist('Sample Proportion: 200', bins=np.arange(0.65, 0.85, 0.01), normed=True)

Histogram with 'Sample Proportion: 200' on the x-axis and 'Percent per unit' on the y-axis. The x-axis ranges from about 0.65 to above 0.8. The data is fairly symmetric and bell shaped, centered around 0.75.

There's that normal curve again, as predicted by the Central Limit Theorem, centered at around 0.75 just as you would expect.

How would this distribution change if we increased the sample size? Let's run the code again with a sample size of 800, and collect the results of simulations in the same DataFrame in which we collected simulations based on a sample size of 200. We will keep the number of `repetitions` the same as before so that the two columns have the same length.

In [23]:
props2 = make_array()

num_plants = 800

for i in np.arange(repetitions):
    sample = model.sample(num_plants, with_replacement=True)
    new_prop = (sample.get_column('Color') == 'Purple').sum()/num_plants
    props2 = np.append(props2, new_prop)

results = results.with_columns(pl.Series('Sample Proportion: 800', props2))

In [24]:
results.hvplot.hist(
    ['Sample Proportion: 200', 'Sample Proportion: 800'],
    bins=np.arange(0.65, 0.85, 0.01),
    normed=True,
    alpha=0.5,
    color=['#002fa7', '#706700'],
)

Two overlaid histograms. Both histograms are bell shaped and symmetrical, centered about 0.75. The wider histogram is in dark blue and labeled 'Sample Proportion: 200'. The wider histogram extends from approximately x=0.67 to 0.81 and has a shorter peak. The less wide histogram is in gold and labeled 'Sample Proportion: 800.' The gold histogram extends from about 0.7 to 0.8 and has a taller peak.

Both distributions are approximately normal but one is narrower than the other. The proportions based on a sample size of 800 are more tightly clustered around 0.75 than those from a sample size of 200. Increasing the sample size has decreased the variability in the sample proportion.

This should not be surprising.  We have leaned many times on the intuition that a larger sample size generally reduces the variability of a statistic.  However, in the case of a sample average, we can *quantify* the relationship between sample size and variability.

Exactly how does the sample size affect the variability of a sample average or proportion? That is the question we will examine in the next section.